# 05. Pressor Raw (승압제 원본 추출)

## 목적
코호트 환자의 승압제 투여 데이터를 **1시간 단위**로 추출

## 입력
- `sepsis_cohort.csv`: 기본 코호트

## 출력
- `pressor_raw.csv`: 시간별 승압제 투여 정보 (1 row = 1 patient × 1 hour)

## 전처리 범위
- [x] 데이터 추출 (inputevents)
- [x] 시간 범위 확장 (start~end → 1시간 단위)
- [x] 용량 정보 추출
- [ ] 슬라이딩 윈도우 적용 → merge 단계에서

## Item IDs
| Item ID | 약물명 |
|---------|--------|
| 221906 | Norepinephrine |
| 221289 | Epinephrine |
| 222315 | Vasopressin |
| 221662 | Dopamine |

In [1]:
from pathlib import Path
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
import duckdb
import pandas as pd
import os

DB_PATH = RAW_DIR / "mimic_total.duckdb"
INPUT_DIR = PROCESSED_DIR
OUTPUT_DIR = PROCESSED_DIR

con = duckdb.connect(DB_PATH)
print("=== 05. Pressor Raw 추출 시작 ===")

=== 05. Pressor Raw 추출 시작 ===


## Step 1: 코호트 로드

In [2]:
print("Step 1: 코호트 로드")

cohort_path = INPUT_DIR / "sepsis_cohort.csv"
df_cohort = pd.read_csv(cohort_path, parse_dates=['intime', 'outtime'])

print(f"✓ 코호트 로드 완료: {len(df_cohort):,}명")

con.execute("CREATE OR REPLACE TEMP TABLE cohort AS SELECT * FROM read_csv_auto(?)", [str(cohort_path)])


Step 1: 코호트 로드
✓ 코호트 로드 완료: 18,001명


## Step 2: Pressor 데이터 추출

In [3]:
print("\nStep 2: Pressor 데이터 추출")

pressor_query = """
WITH pressor_expanded AS (
    -- inputevents의 starttime~endtime을 1시간 단위로 확장
    SELECT
        c.stay_id,
        UNNEST(generate_series(
            date_trunc('hour', CAST(ie.starttime AS TIMESTAMP)),
            date_trunc('hour', COALESCE(CAST(ie.endtime AS TIMESTAMP), CAST(ie.starttime AS TIMESTAMP))),
            INTERVAL '1' HOUR
        )) AS charttime_h,
        ie.itemid,
        TRY_CAST(ie.rate AS DOUBLE) as rate,
        TRY_CAST(ie.amount AS DOUBLE) as amount
    FROM cohort c
    INNER JOIN inputevents ie ON c.stay_id = ie.stay_id
    WHERE ie.itemid IN ('221906', '221289', '222315', '221662')
      AND ie.starttime IS NOT NULL
      AND TRY_CAST(ie.rate AS DOUBLE) > 0
)
SELECT
    stay_id,
    charttime_h,
    1 as pressor_flag,
    
    -- 약물별 사용 여부
    MAX(CASE WHEN itemid = '221906' THEN 1 ELSE 0 END) as norepinephrine_flag,
    MAX(CASE WHEN itemid = '221289' THEN 1 ELSE 0 END) as epinephrine_flag,
    MAX(CASE WHEN itemid = '222315' THEN 1 ELSE 0 END) as vasopressin_flag,
    MAX(CASE WHEN itemid = '221662' THEN 1 ELSE 0 END) as dopamine_flag,
    
    -- 용량 정보 (최대값)
    MAX(CASE WHEN itemid = '221906' THEN rate END) as norepinephrine_rate,
    MAX(CASE WHEN itemid = '221289' THEN rate END) as epinephrine_rate,
    MAX(CASE WHEN itemid = '222315' THEN rate END) as vasopressin_rate,
    MAX(CASE WHEN itemid = '221662' THEN rate END) as dopamine_rate,
    
    -- 동시 투여 약물 수
    (
        MAX(CASE WHEN itemid = '221906' THEN 1 ELSE 0 END) +
        MAX(CASE WHEN itemid = '221289' THEN 1 ELSE 0 END) +
        MAX(CASE WHEN itemid = '222315' THEN 1 ELSE 0 END) +
        MAX(CASE WHEN itemid = '221662' THEN 1 ELSE 0 END)
    ) as pressor_count
    
FROM pressor_expanded
GROUP BY stay_id, charttime_h
ORDER BY stay_id, charttime_h
"""

print("  쿼리 실행 중...")
df_pressor = con.execute(pressor_query).df()

print(f"✓ Pressor 추출 완료")
print(f"  - 총 행 수: {len(df_pressor):,}개 (pressor 투여 중인 시점만)")
print(f"  - 고유 환자: {df_pressor['stay_id'].nunique():,}명")


Step 2: Pressor 데이터 추출
  쿼리 실행 중...
✓ Pressor 추출 완료
  - 총 행 수: 367,081개 (pressor 투여 중인 시점만)
  - 고유 환자: 6,556명


## Step 3: 통계 확인

In [4]:
print("\nStep 3: 통계 확인")

print(f"\n=== Pressor 사용 환자 ===")
print(f"  - 전체: {df_pressor['stay_id'].nunique():,}명 ({df_pressor['stay_id'].nunique() / len(df_cohort) * 100:.1f}%)")

print(f"\n=== 약물별 사용 환자 ===")
for drug in ['norepinephrine', 'epinephrine', 'vasopressin', 'dopamine']:
    flag_col = f'{drug}_flag'
    n_patients = df_pressor[df_pressor[flag_col] == 1]['stay_id'].nunique()
    print(f"  - {drug.capitalize()}: {n_patients:,}명")

print(f"\n=== 동시 투여 약물 수 ===")
print(df_pressor['pressor_count'].value_counts().sort_index())


Step 3: 통계 확인

=== Pressor 사용 환자 ===
  - 전체: 6,556명 (36.4%)

=== 약물별 사용 환자 ===
  - Norepinephrine: 5,800명
  - Epinephrine: 1,257명
  - Vasopressin: 2,110명
  - Dopamine: 602명

=== 동시 투여 약물 수 ===
pressor_count
1    268297
2     87584
3     10719
4       481
Name: count, dtype: int64


## Step 4: 저장

In [5]:
print("\n" + "="*60)
print("CSV 저장")
print("="*60)

output_path = OUTPUT_DIR / "pressor_raw.csv"
df_pressor.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: pressor_raw.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_pressor):,}개")
print(f"  - 경로: {output_path}")


CSV 저장

✓ 저장 완료: pressor_raw.csv
  - 파일 크기: 24.22 MB
  - 행 수: 367,081개
  - 경로: DATA/processed/pressor_raw.csv


In [6]:
print("\n=== 샘플 데이터 ===")
df_pressor.head(10)


=== 샘플 데이터 ===


,stay_id,charttime_h,pressor_flag,norepinephrine_flag,epinephrine_flag,vasopressin_flag,dopamine_flag,norepinephrine_rate,epinephrine_rate,vasopressin_rate,dopamine_rate,pressor_count
0,30001446,2186-04-12 05:00:00,1,0,0,0,1,NaN,NaN,NaN,5.004316,1
1,30001446,2186-04-12 06:00:00,1,0,0,0,1,NaN,NaN,NaN,5.004316,1
2,30001446,2186-04-12 07:00:00,1,0,0,0,1,NaN,NaN,NaN,2.502158,1
3,30001446,2186-04-12 08:00:00,1,0,0,0,1,NaN,NaN,NaN,2.500884,1
4,30001446,2186-04-12 09:00:00,1,1,0,0,1,0.040011,NaN,NaN,2.500884,2
5,30001446,2186-04-12 10:00:00,1,1,0,0,0,0.080021,NaN,NaN,NaN,1
6,30001446,2186-04-12 11:00:00,1,1,0,0,0,0.080021,NaN,NaN,NaN,1
7,30001446,2186-04-12 12:00:00,1,1,0,0,0,0.080021,NaN,NaN,NaN,1
8,30001446,2186-04-12 13:00:00,1,1,0,0,0,0.059994,NaN,NaN,NaN,1
9,30001446,2186-04-12 14:00:00,1,1,0,0,0,0.039981,NaN,NaN,NaN,1


In [7]:
con.close()
print("\n=== 05. Pressor Raw 추출 완료 ===")


=== 05. Pressor Raw 추출 완료 ===
